In [ ]:
import os
import copy
from params.paths import ROOT_DIR
from file_handling.file_read_writer import read_json, write_json
from dbio.representative_db import connect_db, get_person_by_column, get_election_result_by_person_id, get_closest_person_by_name
from utils.string_process import clean_repr_name
SHUGIIN_REPR_LIST_DIR = os.path.join(ROOT_DIR, "data", "data_shugiin", "repr_list")
SANGIIN_REPR_LIST_DIR = os.path.join(ROOT_DIR, "data", "data_sangiin", "repr_list")
SHUGIIN_REPR_LIST_FILE_PATH = os.path.join(SHUGIIN_REPR_LIST_DIR, "20241218_repr_list.json")
SANGIIN_REPR_LIST_FILE_PATH = os.path.join(SANGIIN_REPR_LIST_DIR, "20250911_repr_list.json")

SHUGIIN_REPR_LIST = read_json(SHUGIIN_REPR_LIST_FILE_PATH)["reprs"]
SANGIIN_REPR_LIST = read_json(SANGIIN_REPR_LIST_FILE_PATH)["reprs"]

if any((not os.path.exists(PATH) for PATH in [VISUAL_DATA_DIR, SHUGIIN_REPR_LIST_FILE_PATH, SANGIIN_REPR_LIST_FILE_PATH])):
	raise ValueError("Visual data directory is not found")

print(SHUGIIN_REPR_LIST)
print(SANGIIN_REPR_LIST)


{'自民': [{'name': '逢沢  一郎君', 'yomikata': 'あいさわ  いちろう', 'kaiha': '自民', 'district': '岡山1', 'number_of_terms_lower': 13, 'number_of_terms_upper': 0}, {'name': '赤沢 亮正君', 'yomikata': 'あかざわ  りょうせい', 'kaiha': '自民', 'district': '鳥取2', 'number_of_terms_lower': 7, 'number_of_terms_upper': 0}, {'name': 'あかま 二郎君', 'yomikata': 'あかま  じろう', 'kaiha': '自民', 'district': '神奈川14', 'number_of_terms_lower': 6, 'number_of_terms_upper': 0}, {'name': '東   国幹君', 'yomikata': 'あずま  くによし', 'kaiha': '自民', 'district': '北海道6', 'number_of_terms_lower': 2, 'number_of_terms_upper': 0}, {'name': '麻生  太郎君', 'yomikata': 'あそう  たろう', 'kaiha': '自民', 'district': '福岡8', 'number_of_terms_lower': 15, 'number_of_terms_upper': 0}, {'name': 'あべ  俊子君', 'yomikata': 'あべ  としこ', 'kaiha': '自民', 'district': '（比）九州', 'number_of_terms_lower': 7, 'number_of_terms_upper': 0}, {'name': '安藤 たかお君', 'yomikata': 'あんどう  たかお', 'kaiha': '自民', 'district': '（比）東京', 'number_of_terms_lower': 2, 'number_of_terms_upper': 0}, {'name': '五十嵐  清君', 'yomikata': '

In [2]:
from dataclasses import dataclass
from pydantic import BaseModel
from typing import Optional
from dotenv import load_dotenv

load_dotenv()	



class Representative(BaseModel):
	name:str
	kaiha:str
	district: str
	yomikata: str
	number_of_terms_lower: Optional[int] = None
	number_of_terms_upper: Optional[int] = None
	link: Optional[str] = None
	period: Optional[str] = None
	person_id: Optional[int] = None

	def to_dict(self) -> dict:
		return self.model_dump()
	
	def __str__(self) -> str:
		return f"{self.name} ({self.yomikata})"
	
	def __repr__(self) -> str:
		return self.__str__()
	
	

def iterate_repr_list(repr_list:dict) -> Representative:
	for party in repr_list.keys():
		for repr in repr_list[party]:
			yield Representative(**repr)


conn = connect_db(
    dbname="kokkaidoc",
    user="postgres",
    password=os.getenv("PSQL_DATABASE_PASSWORD"),
    host="localhost",
    port=5432,
)



In [ ]:
target_repr_list = SANGIIN_REPR_LIST
output_repr_list_with_id = []
# output_file_path = os.path.join(SHUGIIN_REPR_LIST_DIR, "20241218_repr_list_with_id.json")
output_file_path = os.path.join(SANGIIN_REPR_LIST_DIR, "20250911_repr_list_with_id.json")
with conn.cursor() as cur:

	for repr in iterate_repr_list(target_repr_list):
		repr_name_clean = clean_repr_name(repr.name)
		print("Working on ", repr.name)
		person = get_person_by_column(cur, "name_kanji", repr_name_clean)
		hiragana_person = get_person_by_column(cur, "name_kana", clean_repr_name(repr.yomikata))
		if len(person) > 1:
			print(f"{repr.name} ({repr.yomikata}) is found in multiple persons.")
			print(f"{repr.kaiha} {repr.district} {repr.number_of_terms_lower} - {repr.number_of_terms_upper}")
			for idx, p in enumerate(person):
				print(f"{idx}: {p.name_kanji} ({p.name_kana})")
				election_result = get_election_result_by_person_id(cur, p.person_id)
				print("\n".join([str(e) for e in election_result]))
		
			selected_idx = int(input(f"{repr.name} ({repr.yomikata}) is found in multiple persons. Please select the correct one: "))
			repr_with_id = copy.deepcopy(repr)
			repr_with_id.person_id = person[int(selected_idx)].person_id
			output_repr_list_with_id.append(repr_with_id)

		elif len(person) == 1:
			repr_with_id = copy.deepcopy(repr)
			repr_with_id.person_id = person[0].person_id
			output_repr_list_with_id.append(repr_with_id)
	
		elif len(hiragana_person) == 1:
			repr_with_id = copy.deepcopy(repr)
			repr_with_id.person_id = hiragana_person[0].person_id
			output_repr_list_with_id.append(repr_with_id)
		else:
			candidates = get_closest_person_by_name(cur, clean_repr_name(repr.name))
			if len(candidates) == 0:
				raise ValueError(f"No person found for {repr.name}")
			for idx, candidate in enumerate(candidates):
				print(idx, candidate)
			selected_idx = int(input(f"{repr.name} ({repr.yomikata}) is not found in the database. Please select the correct one: "))
			repr_with_id = copy.deepcopy(repr)
			repr_with_id.person_id = candidates[int(selected_idx)].person.person_id
			output_repr_list_with_id.append(repr_with_id)

	
			

write_json({"reprs": [repr.to_dict() for repr in output_repr_list_with_id]}, output_file_path)


Working on  青木   愛
Working on  石垣 のりこ [小川 のり子]
Working on  石橋  通宏
Working on  泉   房穂
Working on  打越 さく良 [村木 さく良]
Working on  小沢  雅仁
Working on  鬼木   誠
鬼木   誠 (おにき まこと) is found in multiple persons.
立憲 比例 None - None
0: 鬼木誠 (おにきまこと)
Election date: 2012-12-16 Election name: 第46回衆議院議員総選挙 District: 福岡2区 Party: 自由民主党 Result: 当選 (1)
Election date: 2014-12-14 Election name: 第47回衆議院議員総選挙 District: 福岡2区 Party: 自由民主党 Result: 当選 (2)
Election date: 2017-10-22 Election name: 第48回衆議院議員総選挙 District: 福岡2区 Party: 自由民主党 Result: 当選 (3)
Election date: 2021-10-31 Election name: 第49回衆議院議員総選挙 District: 福岡2区 Party: 自由民主党 Result: 当選 (4)
Election date: 2024-10-27 Election name: 第50回衆議院議員総選挙 District: 九州ブロック Party: 自由民主党 Result: 当選 (5)
1: 鬼木誠 (おにきまこと)
Election date: 2022-07-10 Election name: 第26回参議院議員通常選挙 District: 比例区 Party: 立憲民主党 Result: 当選 (1)
Working on  勝部  賢志
Working on  岸  真紀子
Working on  木戸口 英司
Working on  熊谷  裕人
Working on  郡山 りょう [郡山   玲]
Working on  古賀  千景
Working on  古賀  之士
Working on  小島 とも子 [小島  智子

In [6]:
done_topics = []

In [10]:
import os
import copy
from params.paths import ROOT_DIR
import psycopg2
from file_handling.file_read_writer import read_json, write_json
from dbio.representative_db import connect_db, get_person_by_column, get_election_result_by_person_id, get_closest_person_by_name
from utils.string_process import clean_repr_name
from pydantic import BaseModel
from typing import Optional
from dotenv import load_dotenv

load_dotenv()


class VisualData(BaseModel):
	idx: int
	x: float
	y: float
	repr: str
	hiragana: str
	house: str
	party: str
	color: str
	ref_point: Optional[str] = ''
	person_id: Optional[int] = None

	def to_dict(self) -> dict:
		return self.model_dump()

VISUAL_DATA_DIR = os.path.join(ROOT_DIR, "data", "data_visual", "static")


def get_politician_id_by_name(name_kanji: str, name_kana: str, party: str):
	repr_name_clean = clean_repr_name(name_kanji)
	print("Working on ", name_kanji)
	person = get_person_by_column(cur, "name_kanji", repr_name_clean)
	hiragana_person = get_person_by_column(cur, "name_kana", clean_repr_name(name_kana))
	if len(person) > 1:
		print(f"{name_kanji} ({name_kana}) is found in multiple persons.")
		print(f"{party}")
		for idx, p in enumerate(person):
			print(f"{idx}: {name_kanji} ({name_kana})")
			election_result = get_election_result_by_person_id(cur, p.person_id)
			print("\n".join([str(e) for e in election_result]))
	
		selected_idx = int(input(f"{name_kanji} ({name_kana}) is found in multiple persons. Please select the correct one: "))
		return person[selected_idx].person_id

	elif len(person) == 1:
		return person[0].person_id

	elif len(hiragana_person) == 1:
		return hiragana_person[0].person_id
	else:
		candidates = get_closest_person_by_name(cur, clean_repr_name(name_kanji))
		if len(candidates) == 0:
			raise ValueError(f"No person found for {repr.name}")
		for idx, candidate in enumerate(candidates):
			print(idx, candidate)
		selected_idx = int(input(f"{repr.name} ({repr.yomikata}) is not found in the database. Please select the correct one: "))
		return candidates[int(selected_idx)].person.person_id


def process_visual_data(file_path: str, output_file_path: str, cur: psycopg2.extensions.cursor, ):
	data = read_json(file_path)
	data_with_id = []
	for item in data["data"]:
		item_obj = VisualData(**item)
		if item_obj.repr == "反対" or item_obj.repr == "賛成":
			data_with_id.append(item_obj.to_dict())
			continue
		politician_id = get_politician_id_by_name(item_obj.repr, item_obj.hiragana, item_obj.party)
		item_obj.person_id = politician_id
		data_with_id.append(item_obj.to_dict())
	
	write_json({"data": data_with_id}, output_file_path)




try:
	conn = connect_db(
	dbname="kokkaidoc",
	user="postgres",
	password=os.getenv("PSQL_DATABASE_PASSWORD"),
	host="localhost",
	port=5432,
	)

	cur = conn.cursor()
	for topic in os.listdir(VISUAL_DATA_DIR):
		
		topic_dir = os.path.join(VISUAL_DATA_DIR, topic)
		if os.path.isfile(topic_dir):
			continue
		for sub_topic in os.listdir(topic_dir):
			if f"{topic} {sub_topic}" in done_topics:
				print(f"Skipping {topic} {sub_topic} because it has already been processed.")
				continue
			print("-----------------------------------------------")
			print(f"Processing {topic}")
			print(f"Processing {sub_topic}")
			print("-----------------------------------------------")
			file_1d = os.path.join(topic_dir, sub_topic, "gen_1d.json")
			file_2d = os.path.join(topic_dir, sub_topic, "gen_2d.json")
			for dim, file in enumerate([file_1d, file_2d]):
				process_visual_data(file_path=file, 
					output_file_path=file.replace(".json", "_with_id.json"), 
					cur=cur)
			done_topics.append(f"{topic} {sub_topic}")

except Exception as e:
	raise e

finally:
	cur.close()



Skipping nuclear export because it has already been processed.
Skipping nuclear newbuildandextendlifetime because it has already been processed.
Skipping nuclear abolishandrenewable because it has already been processed.
Skipping nuclear restart because it has already been processed.
Skipping nuclear fukushimasupport because it has already been processed.
Skipping pension employeespensionissue because it has already been processed.
Skipping pension pensionsystemunification because it has already been processed.
Skipping pension basicpensionbottomup because it has already been processed.
Skipping pension pensionreformtransparency because it has already been processed.
Skipping pension basicpensionsubsidy because it has already been processed.
Skipping socialsecurity sustainability because it has already been processed.
Skipping socialsecurity younggenerationburden because it has already been processed.
Skipping socialsecurity benefitburdenbalance because it has already been processed.
S

## Combine all the visual data into one file

In [ ]:
import os
from params.paths import ROOT_DIR
from file_handling.file_read_writer import read_json, write_json
from dotenv import load_dotenv


data_visual_dir = os.path.join(ROOT_DIR, "data", "data_visual", "static")

all_data = []
for topic in os.listdir(data_visual_dir):
	topic_dir = os.path.join(data_visual_dir, topic)
	if os.path.isfile(topic_dir):
		continue
	topic_data = {
		"topic": topic,
		"sub_topics": []
	}
	for sub_topic in os.listdir(topic_dir):
		sub_topic_dir = os.path.join(topic_dir, sub_topic)
		file_1d = os.path.join(sub_topic_dir, "gen_1d_with_id.json")
		file_2d = os.path.join(sub_topic_dir, "gen_2d_with_id.json")
		data_1d = read_json(file_1d)
		data_2d = read_json(file_2d)
		sub_topic_data = {
			"topic": topic,
			"sub_topic": sub_topic,
			"1d": data_1d,
			"2d": data_2d
		}
		topic_data["sub_topics"].append(sub_topic_data)
	all_data.append(topic_data)

write_json({"data": all_data}, os.path.join(data_visual_dir, "all_data.json"))
